<a href="https://colab.research.google.com/github/MuhammadAmaan178/Demand-Lens/blob/main/notebooks/DLens_09_Inventory_%26_Reorder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Step 1: Import Libraries and Verify Uploaded Files**

In [10]:
import os
import json
import math
import numpy as np
import pandas as pd
import joblib

In [11]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [12]:
# Use the existing Google Drive export folder
EXPORT_DIR = "/content/drive/MyDrive/DemandLens/Data/processed"

In [13]:
# Define paths for all 7 files
MODEL_H7_PATH = os.path.join(EXPORT_DIR, "lightgbm_h7.joblib")
MODEL_H28_PATH = os.path.join(EXPORT_DIR, "lightgbm_h28.joblib")

VAL_H7_PATH = os.path.join(EXPORT_DIR, "ca1_h7_val.parquet")
VAL_H28_PATH = os.path.join(EXPORT_DIR, "ca1_h28_val.parquet")

FEATURES_PATH = os.path.join(EXPORT_DIR, "feature_lists.json")
H7_VAL_PREDS_PATH = os.path.join(EXPORT_DIR, "h7_val_predictions.parquet")
ITEM_GROUPS_PATH = os.path.join(EXPORT_DIR, "item_groups.csv")

In [14]:
# Store all file paths in one dictionary
files = {
    "H7 LightGBM Model": MODEL_H7_PATH,
    "H28 LightGBM Model": MODEL_H28_PATH,
    "H7 Validation Data": VAL_H7_PATH,
    "H28 Validation Data": VAL_H28_PATH,
    "Feature Lists": FEATURES_PATH,
    "H7 Validation Predictions": H7_VAL_PREDS_PATH,
    "Item Groups": ITEM_GROUPS_PATH
}

In [15]:
# Check whether all files exist
print("Checking Google Drive files...\n")

all_found = True

for name, path in files.items():

    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"[OK] {name}: {size:.2f} MB")

    else:
        print(f"[MISSING] {name}: File not found")
        all_found = False


# Display final status
if all_found:
    print("\nAll 7 files are ready!")
else:
    print("\nSome files are missing. Please check your Google Drive folder.")

Checking Google Drive files...

[OK] H7 LightGBM Model: 1.74 MB
[OK] H28 LightGBM Model: 8.33 MB
[OK] H7 Validation Data: 0.22 MB
[OK] H28 Validation Data: 0.23 MB
[OK] Feature Lists: 0.06 MB
[OK] H7 Validation Predictions: 0.78 MB
[OK] Item Groups: 0.30 MB

All 7 files are ready!


## **Step 2: Load Models and Datasets & Inspect Columns**

### **What we are doing**

We are loading both pre-trained LightGBM models (H7 and H28), feature lists, validation predictions, item groups, and both validation datasets.

We will inspect their shapes, column names, date ranges, and first few rows.

### **Why we need this step**

Before building our inventory and reorder calculations, we need to understand the actual dataset structure, especially the prediction and H28 validation columns.

### **What problem this solves**

It helps prevent `KeyError` issues caused by incorrect column names and confirms that the required 3,049 items are available.

### **Expected output**

- Confirmation that both models are loaded.
- Shapes and column names of all datasets.
- Date ranges of validation data.
- First 5 rows of H7 predictions and item groups.

In [16]:
# Step 2: Load Models and Datasets & Inspect Columns

# 1. Load both pre-trained LightGBM models
model_h7 = joblib.load(MODEL_H7_PATH)
model_h28 = joblib.load(MODEL_H28_PATH)

print("Both models loaded successfully.")
print("H7 Model:", type(model_h7))
print("H28 Model:", type(model_h28))

Both models loaded successfully.
H7 Model: <class 'lightgbm.sklearn.LGBMRegressor'>
H28 Model: <class 'lightgbm.sklearn.LGBMRegressor'>


In [17]:
# 2. Load feature lists from JSON file
with open(FEATURES_PATH, "r") as file:
    feature_lists = json.load(file)

print("Feature List Keys:", list(feature_lists.keys()))

Feature List Keys: ['features_7', 'features_28', 'item_categories']


In [18]:
# 3. Load all required datasets
h7_val_preds = pd.read_parquet(H7_VAL_PREDS_PATH)
item_groups = pd.read_csv(ITEM_GROUPS_PATH)

val_h7_df = pd.read_parquet(VAL_H7_PATH)
val_h28_df = pd.read_parquet(VAL_H28_PATH)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [19]:
# 4. Check the shape and columns of each dataset
print("H7 Predictions Shape:", h7_val_preds.shape)
print("H7 Predictions Columns:", h7_val_preds.columns.tolist())

print("Item Groups Shape:", item_groups.shape)
print("Item Groups Columns:", item_groups.columns.tolist())

print("H7 Validation Shape:", val_h7_df.shape)
print("H7 Validation Columns:", val_h7_df.columns.tolist())

print("H28 Validation Shape:", val_h28_df.shape)
print("H28 Validation Columns:", val_h28_df.columns.tolist())

H7 Predictions Shape: (85372, 4)
H7 Predictions Columns: ['item_id', 'date', 'sales', 'forecast']
Item Groups Shape: (3049, 6)
Item Groups Columns: ['item_id', 'avg_sales', 'zero_sales_share', 'sales_std', 'weekend_sales_ratio', 'group']
H7 Validation Shape: (85372, 26)
H7 Validation Columns: ['item_id', 'store_id', 'date', 'sales', 'lag_7', 'rolling_mean_7', 'wday', 'month', 'year', 'is_weekend', 'snap', 'has_event', 'sell_price', 'price_change', 'ev_Cultural', 'ev_National', 'ev_Religious', 'ev_Sporting', 'ev_None', 'dept_FOODS_1', 'dept_FOODS_2', 'dept_FOODS_3', 'dept_HOBBIES_1', 'dept_HOBBIES_2', 'dept_HOUSEHOLD_1', 'dept_HOUSEHOLD_2']
H28 Validation Shape: (85359, 26)
H28 Validation Columns: ['item_id', 'store_id', 'date', 'sales', 'lag_28', 'rolling_mean_28', 'wday', 'month', 'year', 'is_weekend', 'snap', 'has_event', 'sell_price', 'price_change', 'ev_Cultural', 'ev_National', 'ev_Religious', 'ev_Sporting', 'ev_None', 'dept_FOODS_1', 'dept_FOODS_2', 'dept_FOODS_3', 'dept_HOBBIES_

In [20]:
# 5. Check date ranges
if "date" in h7_val_preds.columns:
    print("H7 Predictions Date Range:")
    print(h7_val_preds["date"].min(), "to", h7_val_preds["date"].max())

if "date" in val_h28_df.columns:
    print("H28 Validation Date Range:")
    print(val_h28_df["date"].min(), "to", val_h28_df["date"].max())

H7 Predictions Date Range:
2016-03-28 00:00:00 to 2016-04-24 00:00:00
H28 Validation Date Range:
2016-03-28 00:00:00 to 2016-04-24 00:00:00


In [21]:
# 6. Check missing values
print("Missing Values in H7 Predictions:", h7_val_preds.isnull().sum().sum())
print("Missing Values in Item Groups:", item_groups.isnull().sum().sum())
print("Missing Values in H7 Validation:", val_h7_df.isnull().sum().sum())
print("Missing Values in H28 Validation:", val_h28_df.isnull().sum().sum())

Missing Values in H7 Predictions: 0
Missing Values in Item Groups: 0
Missing Values in H7 Validation: 0
Missing Values in H28 Validation: 0


In [22]:
# 7. Display the first 5 rows of important datasets
print("H7 Validation Predictions:")
display(h7_val_preds.head())

print("Item Groups:")
display(item_groups.head())

H7 Validation Predictions:


,item_id,date,sales,forecast
0,FOODS_1_001,2016-03-28,2,0.779472
1,FOODS_1_001,2016-03-29,1,0.888189
2,FOODS_1_001,2016-03-30,1,0.994175
3,FOODS_1_001,2016-03-31,0,0.805700
4,FOODS_1_001,2016-04-01,4,1.101319


Item Groups:


,item_id,avg_sales,zero_sales_share,sales_std,weekend_sales_ratio,group
0,FOODS_1_001,0.780579,0.565451,1.258201,0.355326,Slow Sellers
1,FOODS_1_002,0.480945,0.667203,0.807814,0.314732,Slow Sellers
2,FOODS_1_003,0.837634,0.527957,1.169582,0.318357,Slow Sellers
3,FOODS_1_004,10.672367,0.123119,8.629497,0.351855,Fast Sellers
4,FOODS_1_005,1.156116,0.541309,2.019971,0.349420,Slow Sellers


### **Step 3: Calculate Typical Daily Forecast Error (**`item_error_spread`**)**

**What we are doing**

We calculate the forecast error for each row by subtracting the forecast from actual sales.

Then, we calculate the standard deviation of these errors for each of the 3,049 items.

If an item has fewer than 7 validation days or its standard deviation is missing, we use the average error spread of its item group instead.

**Why we need this step**

Forecasts are not always accurate. We need to measure how much forecast errors usually vary for each item so we can calculate the required Safety Stock.

**What problem this step solves**

It gives every item a reliable measure of forecast uncertainty (`typical_daily_error`), even if some items have limited validation data.

**Expected output**

- Total number of items processed (3,049).
- Number of items that used the group-average fallback.
- Average typical daily error for each item group.
- First 5 rows of `item_error_spread`.

In [23]:
# Step 3: Calculate Typical Daily Forecast Error

# 1. Calculate the difference between actual sales and forecast
h7_val_preds["error"] = h7_val_preds["sales"] - h7_val_preds["forecast"]

In [24]:
# 2. Calculate the number of validation days and error standard deviation for each item
item_error_stats = (
    h7_val_preds.groupby("item_id")["error"]
    .agg(["count", "std"])
    .reset_index()
)

item_error_stats.columns = ["item_id", "val_rows", "raw_error_std"]

In [25]:
# 3. Add each item's K-Means group
item_error_spread = item_groups[["item_id", "group"]].merge(
    item_error_stats,
    on="item_id",
    how="left"
)

# Replace missing validation counts with 0
item_error_spread["val_rows"] = item_error_spread["val_rows"].fillna(0).astype(int)

In [26]:
# 4. Calculate the average error standard deviation for each group
MIN_VAL_ROWS = 7

valid_items = item_error_spread[
    item_error_spread["val_rows"] >= MIN_VAL_ROWS
]

group_avg_error = valid_items.groupby("group")["raw_error_std"].mean()

print("Average Daily Forecast Error by Item Group:")

for group, avg_error in group_avg_error.items():
    print(f"{group}: {avg_error:.4f} units/day")

Average Daily Forecast Error by Item Group:
Fast Sellers: 7.0805 units/day
Medium Sellers: 2.1052 units/day
Slow Sellers: 0.7967 units/day


In [27]:
# 5. Use the group average when an item has insufficient data
needs_fallback = (
    (item_error_spread["val_rows"] < MIN_VAL_ROWS) |
    (item_error_spread["raw_error_std"].isna())
)

item_error_spread["used_group_fallback"] = needs_fallback

item_error_spread["typical_daily_error"] = np.where(
    needs_fallback,
    item_error_spread["group"].map(group_avg_error),
    item_error_spread["raw_error_std"]
)

In [28]:
# 6. Display the final results
print("Total Items:", len(item_error_spread))
print("Items using group average:", needs_fallback.sum())
print("Missing Typical Daily Errors:", item_error_spread["typical_daily_error"].isna().sum())

print("First 5 Items:")
display(item_error_spread.head())

Total Items: 3049
Items using group average: 0
Missing Typical Daily Errors: 0
First 5 Items:


,item_id,group,val_rows,raw_error_std,used_group_fallback,typical_daily_error
0,FOODS_1_001,Slow Sellers,28,1.290134,False,1.290134
1,FOODS_1_002,Slow Sellers,28,0.806307,False,0.806307
2,FOODS_1_003,Slow Sellers,28,1.057344,False,1.057344
3,FOODS_1_004,Fast Sellers,28,0.821892,False,0.821892
4,FOODS_1_005,Slow Sellers,28,1.207497,False,1.207497


### **Step 4: Generate H28 Validation Forecasts (For Lead Times Over 7 Days)**

**What we are doing**

We prepare `val_h28_df` by creating the four required cyclical time features:

- `wday_sin`
- `wday_cos`
- `month_sin`
- `month_cos`

We also set `item_id` using the exact category list stored in `feature_lists.json`.

Then, we use the pre-trained `model_h28` to generate 28-day forecasts and save the predictions in `h28_val_preds`.

**Why we need this step**

DemandLens uses:

- H7 forecasts when supplier lead time is 7 days or less.
- H28 forecasts when supplier lead time is more than 7 days.

We already have H7 predictions. Now we need H28 predictions to support longer supplier lead times.

**What problem this step solves**

It allows our reorder system to handle both short and long supplier lead times without training another model.

**Expected output**

- Confirmation that all required H28 model features are available.
- H28 validation WAPE score.
- Shape of `h28_val_preds`.
- First 5 rows of the H28 predictions.

In [29]:
# Step 4: Generate H28 Validation Forecasts

# 1. Create the four cyclical time features
val_h28_df["wday_sin"] = np.sin(2 * np.pi * val_h28_df["wday"] / 7)
val_h28_df["wday_cos"] = np.cos(2 * np.pi * val_h28_df["wday"] / 7)

val_h28_df["month_sin"] = np.sin(2 * np.pi * val_h28_df["month"] / 12)
val_h28_df["month_cos"] = np.cos(2 * np.pi * val_h28_df["month"] / 12)

In [30]:
# 2. Set item_id categories to match the training data
val_h28_df["item_id"] = pd.Categorical(
    val_h28_df["item_id"],
    categories=feature_lists["item_categories"]
)

In [31]:


# 3. Check that all required model features are available
features_28 = list(model_h28.feature_name_)

missing_h28 = [
    col for col in features_28
    if col not in val_h28_df.columns
]

if missing_h28:
    print("Missing features:", missing_h28)

else:
    print("All required H28 model features are available.")

    # Generate forecasts using the pre-trained H28 model
    print("Generating H28 forecasts...")

    raw_h28_preds = model_h28.predict(val_h28_df[features_28])

    # Replace negative forecasts with 0
    val_h28_df["forecast"] = np.clip(raw_h28_preds, 0, None)


    # Create a clean table with actual sales and forecasts
    h28_val_preds = val_h28_df[
        ["item_id", "date", "sales", "forecast"]
    ].copy()


    # Calculate H28 validation WAPE
    h28_abs_err = np.abs(
        h28_val_preds["sales"] - h28_val_preds["forecast"]
    ).sum()

    h28_wape = (
        h28_abs_err / h28_val_preds["sales"].sum()
    ) * 100

    print(f"H28 Validation WAPE: {h28_wape:.2f}%")

    print("H28 Predictions Shape:", h28_val_preds.shape)

    # Display the first 5 predictions
    display(h28_val_preds.head())

All required H28 model features are available.
Generating H28 forecasts...
H28 Validation WAPE: 74.33%
H28 Predictions Shape: (85359, 4)


,item_id,date,sales,forecast
0,FOODS_1_001,2016-03-28,2,0.648588
1,FOODS_1_001,2016-03-29,1,0.557860
2,FOODS_1_001,2016-03-30,1,0.641618
3,FOODS_1_001,2016-03-31,0,0.668558
4,FOODS_1_001,2016-04-01,4,0.680063


### **Step 5: Build the Core Inventory & Reorder Calculation Function**

**What we are doing**

We create a reusable Python function called `calculate_reorder_plan()` that takes three inputs:

- `item_id`: The item we want to check.
- `current_stock`: The quantity currently available.
- `lead_time_days`: How many days the supplier takes to deliver.

The function calculates six important inventory metrics:

1. **Expected Demand:** Uses H7 forecasts for lead times of 7 days or less, and H28 forecasts for lead times over 7 days.
2. **Safety Stock:** Extra stock kept to protect against unexpected demand or forecast errors.
3. **Reorder Point:** The stock level at which we should place a new order.
4. **Days of Stock Left:** Estimates how many days the current stock will last.
5. **Order Quantity:** Calculates how much stock we should order, including the 7-day review period.
6. **Stockout Risk:** Classifies the risk as High, Medium, or Low based on how long the current stock will last compared to supplier lead time.

**Why we need this step**

We put all inventory calculations into one reusable function so that the same logic can be used throughout the notebook and later in the DemandLens backend.

**What problem this step solves**

It converts sales forecasts into practical inventory decisions, helping us understand when to reorder, how much to order, and how urgently an item needs attention.

It also handles cases where forecasted demand is zero.

**Expected output**

A confirmation message showing that `calculate_reorder_plan()` has been successfully defined and is ready for testing.

In [32]:
# Step 5: Build the Inventory and Reorder Calculation Function

def calculate_reorder_plan(
    item_id,
    current_stock,
    lead_time_days,
    h7_preds_df,
    h28_preds_df,
    error_spread_df,
    review_days=7
):

    # 1. Select H7 or H28 forecasts based on supplier lead time
    if lead_time_days <= 7:
        forecast_source = "H7"
        item_forecasts = h7_preds_df[h7_preds_df["item_id"] == item_id]
    else:
        forecast_source = "H28"
        item_forecasts = h28_preds_df[h28_preds_df["item_id"] == item_id]

    item_forecasts = item_forecasts.sort_values("date")

    if item_forecasts.empty:
        raise ValueError(f"Item '{item_id}' not found in {forecast_source} data.")

    daily_forecasts = item_forecasts["forecast"].values

    # 2. Calculate expected demand during supplier lead time
    lead_time_forecasts = daily_forecasts[:lead_time_days]

    expected_demand_lead_time = float(np.sum(lead_time_forecasts))
    avg_daily_forecast = float(np.mean(lead_time_forecasts))

    # 3. Get the item's group and typical daily forecast error
    item_error = error_spread_df[
        error_spread_df["item_id"] == item_id
    ]

    if item_error.empty:
        raise ValueError(f"Item '{item_id}' not found in error spread data.")

    typical_daily_error = float(item_error["typical_daily_error"].iloc[0])
    item_group = str(item_error["group"].iloc[0])

    # 4. Calculate safety stock and reorder point
    safety_stock = 1.65 * typical_daily_error * math.sqrt(lead_time_days)

    reorder_point = expected_demand_lead_time + safety_stock

    # 5. Estimate how many days the current stock will last
    if avg_daily_forecast > 0:
        days_of_stock_left = current_stock / avg_daily_forecast
    else:
        days_of_stock_left = 999.0 if current_stock > 0 else 0.0

    # 6. Calculate the required order quantity
    total_window_days = lead_time_days + review_days

    if len(daily_forecasts) >= total_window_days:
        expected_demand_window = np.sum(daily_forecasts[:total_window_days])
    else:
        missing_days = total_window_days - len(daily_forecasts)

        expected_demand_window = (
            np.sum(daily_forecasts) + missing_days * avg_daily_forecast
        )

    raw_order_qty = expected_demand_window + safety_stock - current_stock

    order_quantity = max(0, math.ceil(raw_order_qty))

    # 7. Classify stockout risk
    if days_of_stock_left < lead_time_days:
        risk = "High"
    elif days_of_stock_left < lead_time_days + 3:
        risk = "Medium"
    else:
        risk = "Low"

    # 8. Return all inventory calculations
    return {
        "item_id": item_id,
        "group": item_group,
        "forecast_model_used": forecast_source,
        "current_stock": current_stock,
        "lead_time_days": lead_time_days,
        "average_daily_forecast": round(avg_daily_forecast, 2),
        "expected_demand_in_lead_time": round(expected_demand_lead_time, 2),
        "typical_daily_error": round(typical_daily_error, 2),
        "safety_stock": round(safety_stock, 2),
        "reorder_point": round(reorder_point, 2),
        "days_of_stock_left": round(days_of_stock_left, 1),
        "order_quantity": int(order_quantity),
        "risk": risk
    }


print("calculate_reorder_plan() is defined and ready to test!")

calculate_reorder_plan() is defined and ready to test!


### **Step 6: Test the Reorder Calculation Function on Sample Scenarios**

**What we are doing**

We test `calculate_reorder_plan()` using four example scenarios.

These scenarios cover:

- Fast, Medium, and Slow Sellers.
- Short supplier lead times (7 days or less), which use H7 forecasts.
- Long supplier lead times (more than 7 days), which use H28 forecasts.
- Different stock levels to test High, Medium, and Low stockout risks.

*Note: All stock quantities and lead times are sample inputs for testing only. They are not actual store inventory values.*

**Why we need this step**

Before exporting the function into `reorder.py` for the DemandLens backend, we need to make sure all calculations work correctly.

We especially want to check:

- Safety Stock calculations.
- Order Quantity and rounding.
- Automatic switching between H7 and H28 models.
- Stockout Risk classification.

**What problem this step solves**

It helps us find calculation errors, unexpected results, and division-by-zero problems before integrating the function into the backend.

**Expected output**

A summary table showing the results for all four test scenarios, including:

- Item ID and seller group.
- Forecast model used (H7 or H28).
- Safety Stock.
- Reorder Point.
- Days of Stock Left.
- Order Quantity.
- Stockout Risk.

In [33]:
# Pick one sample item from each seller group
fast_item = item_error_spread[item_error_spread["group"] == "Fast Sellers"]["item_id"].iloc[0]
medium_item = item_error_spread[item_error_spread["group"] == "Medium Sellers"]["item_id"].iloc[0]
slow_item = item_error_spread[item_error_spread["group"] == "Slow Sellers"]["item_id"].iloc[0]

In [34]:
# Create four test scenarios with different stock levels and lead times
test_scenarios = [
    {"item_id": fast_item, "current_stock": 10, "lead_time_days": 5},
    {"item_id": medium_item, "current_stock": 20, "lead_time_days": 7},
    {"item_id": slow_item, "current_stock": 50, "lead_time_days": 10},
    {"item_id": fast_item, "current_stock": 40, "lead_time_days": 14}
]

In [35]:
# Calculate the reorder plan for each scenario
results_list = []

for scenario in test_scenarios:
    plan = calculate_reorder_plan(
        item_id=scenario["item_id"],
        current_stock=scenario["current_stock"],
        lead_time_days=scenario["lead_time_days"],
        h7_preds_df=h7_val_preds,
        h28_preds_df=h28_val_preds,
        error_spread_df=item_error_spread
    )

    results_list.append(plan)

In [36]:
# Display the results in a DataFrame
test_results_df = pd.DataFrame(results_list)

print("Reorder Intelligence Test Results (Example Inputs)")
display(test_results_df)

Reorder Intelligence Test Results (Example Inputs)


,item_id,group,forecast_model_used,current_stock,lead_time_days,average_daily_forecast,expected_demand_in_lead_time,typical_daily_error,safety_stock,reorder_point,days_of_stock_left,order_quantity,risk
0,FOODS_1_004,Fast Sellers,H7,10,5,2.26,11.32,0.82,3.03,14.35,4.4,26,High
1,FOODS_1_006,Medium Sellers,H7,20,7,1.40,9.83,1.75,7.62,17.45,14.2,8,Low
2,FOODS_1_001,Slow Sellers,H28,50,10,0.64,6.43,1.29,6.73,13.16,77.8,0,Low
3,FOODS_1_004,Fast Sellers,H28,40,14,1.47,20.63,0.82,5.07,25.71,27.1,0,Low


### **Step 7: Export&#x20;**`item_error_spread.csv`**&#x20;and&#x20;**`reorder.py`**&#x20;to Google Drive**

**What we are doing**

We save two important files to the DemandLens Google Drive folder:

- `item_error_spread.csv`: Contains the item groups and typical daily forecast error for all 3,049 items.
- `reorder.py`: Contains the reusable `calculate_reorder_plan()` function, which can be imported into other Python scripts or the backend.

**Why we need this step**

Colab's temporary storage can be cleared. Saving these files to Google Drive keeps them available for future use and backend integration.

**What problem this step solves**

It separates our inventory calculation logic from the notebook and packages it into a standalone Python module that can be reused in the DemandLens backend.

**Expected output**

- Confirmation that Google Drive is mounted.
- `[SAVED]` confirmation for `item_error_spread.csv`.
- `[SAVED]` confirmation for `reorder.py`.
- File sizes for both saved files.

**Important:** Since `EXPORT_DIR` is already defined as your DemandLens processed-data folder, use that existing path instead of mounting Google Drive again or creating another `backend_handover` folder.

In [37]:
# Save both files in the existing DemandLens Google Drive folder
os.makedirs(EXPORT_DIR, exist_ok=True)


# Export item_error_spread.csv
error_spread_path = os.path.join(EXPORT_DIR, "item_error_spread.csv")

item_error_spread.to_csv(error_spread_path, index=False)


# Create the standalone reorder.py module
reorder_module_code = '''"""
DemandLens - Inventory and Reorder Intelligence Module
Store CA_1
"""

import math
import numpy as np


def calculate_reorder_plan(
    item_id,
    current_stock,
    lead_time_days,
    h7_preds_df,
    h28_preds_df,
    error_spread_df,
    review_days=7
):
    """Calculate inventory and reorder intelligence."""

    # Select H7 or H28 forecasts based on lead time
    if lead_time_days <= 7:
        forecast_source = "H7"
        item_forecasts = h7_preds_df[h7_preds_df["item_id"] == item_id]
    else:
        forecast_source = "H28"
        item_forecasts = h28_preds_df[h28_preds_df["item_id"] == item_id]

    item_forecasts = item_forecasts.sort_values("date")

    if item_forecasts.empty:
        raise ValueError(f"Item '{item_id}' not found in {forecast_source} data.")

    daily_forecasts = item_forecasts["forecast"].values

    # Calculate expected demand during lead time
    lead_time_slice = daily_forecasts[:lead_time_days]

    expected_demand_lead_time = float(np.sum(lead_time_slice))
    avg_daily_forecast = float(np.mean(lead_time_slice))

    # Get the item's group and typical daily error
    item_error = error_spread_df[error_spread_df["item_id"] == item_id]

    if item_error.empty:
        raise ValueError(f"Item '{item_id}' not found in item_error_spread.")

    typical_daily_error = float(item_error["typical_daily_error"].iloc[0])
    item_group = str(item_error["group"].iloc[0])

    # Calculate safety stock and reorder point
    safety_stock = 1.65 * typical_daily_error * math.sqrt(lead_time_days)
    reorder_point = expected_demand_lead_time + safety_stock

    # Estimate how many days the current stock will last
    if avg_daily_forecast > 0:
        days_of_stock_left = current_stock / avg_daily_forecast
    else:
        days_of_stock_left = 999.0 if current_stock > 0 else 0.0

    # Calculate the required order quantity
    total_window_days = lead_time_days + review_days

    if len(daily_forecasts) >= total_window_days:
        expected_demand_window = float(np.sum(daily_forecasts[:total_window_days]))
    else:
        missing_days = total_window_days - len(daily_forecasts)
        expected_demand_window = (
            float(np.sum(daily_forecasts)) + missing_days * avg_daily_forecast
        )

    raw_order_qty = expected_demand_window + safety_stock - current_stock
    order_quantity = max(0, math.ceil(raw_order_qty))

    # Classify stockout risk
    if days_of_stock_left < lead_time_days:
        risk = "High"
    elif days_of_stock_left < lead_time_days + 3:
        risk = "Medium"
    else:
        risk = "Low"

    # Return the complete reorder plan
    return {
        "item_id": item_id,
        "group": item_group,
        "forecast_model_used": forecast_source,
        "current_stock": current_stock,
        "lead_time_days": lead_time_days,
        "average_daily_forecast": round(avg_daily_forecast, 2),
        "expected_demand_in_lead_time": round(expected_demand_lead_time, 2),
        "typical_daily_error": round(typical_daily_error, 2),
        "safety_stock": round(safety_stock, 2),
        "reorder_point": round(reorder_point, 2),
        "days_of_stock_left": round(days_of_stock_left, 1),
        "order_quantity": int(order_quantity),
        "risk": risk
    }
'''


# Save reorder.py to Google Drive
reorder_py_path = os.path.join(EXPORT_DIR, "reorder.py")

with open(reorder_py_path, "w") as file:
    file.write(reorder_module_code)


# Verify that both files were saved
print("Google Drive Export Location:")
print(EXPORT_DIR)

print("Export Verification:")

for filepath in [error_spread_path, reorder_py_path]:
    size_kb = os.path.getsize(filepath) / 1024
    print(f"[SAVED] {os.path.basename(filepath)} ({size_kb:.1f} KB)")

print("Both files exported successfully!")

Google Drive Export Location:
/content/drive/MyDrive/DemandLens/Data/processed
Export Verification:
[SAVED] item_error_spread.csv (221.3 KB)
[SAVED] reorder.py (3.2 KB)
Both files exported successfully!


## **Step 8: Update&#x20;**`reorder.py`**&#x20;with&#x20;**`reorder_advice()`**&#x20;and Input Validation**

### **What we are doing**

We are updating the existing `reorder.py` file in our Google Drive folder:

`DemandLens/Data/processed`

We will make these changes:

- Add a new function named `reorder_advice()`.
- Keep `calculate_reorder_plan()` as a backward-compatible alias so existing code continues to work.
- Add input validation to reject:
  - Negative stock (`current_stock < 0`).
  - Zero or negative lead time (`lead_time_days <= 0`).
  - Missing or invalid `item_id`.
- Keep all existing inventory formulas unchanged.
- Leave `item_error_spread.csv` and all other files untouched.

### **Why we are doing this**

Our previous verification found two problems:

- The required function name `reorder_advice()` was missing.
- Invalid inputs, such as negative stock and zero lead time, were not properly checked.

We need to fix these before connecting the module to the backend.

### **Verification tests**

After updating the file, we will import `reorder_advice()` directly from Google Drive and run six tests:

| Test                | Expected result                         |
| ------------------- | --------------------------------------- |
| Normal reorder case | Valid reorder plan                      |
| Zero stock          | Valid plan with required order quantity |
| Zero demand         | No division error                       |
| Negative stock      | Reject input                            |
| Zero lead time      | Reject input                            |
| Invalid `item_id`   | Reject input                            |

### **Expected output**

- `reorder.py` updated successfully.
- `reorder_advice()` imported successfully.
- All six verification tests display `[PASS]`.

**Important path correction:** We will use your existing `EXPORT_DIR` (`DemandLens/Data/processed`), not create or use a separate `backend_handover` folder.

In [39]:
import os
import sys
import importlib


# Save reorder.py in the existing DemandLens Google Drive folder
os.makedirs(EXPORT_DIR, exist_ok=True)

reorder_py_path = os.path.join(EXPORT_DIR, "reorder.py")


# Updated reorder.py module
updated_reorder_code = '''"""
DemandLens - Inventory and Reorder Intelligence Module
Store CA_1
"""

import math
import numbers
import numpy as np


def reorder_advice(
    item_id,
    current_stock,
    lead_time_days,
    h7_preds_df,
    h28_preds_df,
    error_spread_df,
    review_days=7
):
    """Calculate inventory and reorder advice for one item."""

    # Validate item ID
    if not isinstance(item_id, str) or not item_id.strip():
        raise ValueError("Invalid item_id: must be a non-empty string.")

    # Validate current stock
    if (
        isinstance(current_stock, bool)
        or not isinstance(current_stock, numbers.Real)
        or not math.isfinite(current_stock)
        or current_stock < 0
    ):
        raise ValueError("Invalid current_stock: must be a number >= 0.")

    # Validate lead time
    if (
        isinstance(lead_time_days, bool)
        or not isinstance(lead_time_days, int)
        or lead_time_days <= 0
    ):
        raise ValueError("Invalid lead_time_days: must be a positive integer.")

    # Select H7 or H28 forecasts
    if lead_time_days <= 7:
        forecast_source = "H7"
        item_forecasts = h7_preds_df[
            h7_preds_df["item_id"] == item_id
        ]
    else:
        forecast_source = "H28"
        item_forecasts = h28_preds_df[
            h28_preds_df["item_id"] == item_id
        ]

    item_forecasts = item_forecasts.sort_values("date")

    if item_forecasts.empty:
        raise ValueError(
            f"Item '{item_id}' not found in {forecast_source} forecast data."
        )

    daily_forecasts = item_forecasts["forecast"].values

    # Calculate expected demand during lead time
    lead_time_slice = daily_forecasts[:lead_time_days]

    expected_demand_lead_time = float(np.sum(lead_time_slice))
    avg_daily_forecast = float(np.mean(lead_time_slice))

    # Get the item's error information
    item_error = error_spread_df[
        error_spread_df["item_id"] == item_id
    ]

    if item_error.empty:
        raise ValueError(
            f"Item '{item_id}' not found in item_error_spread."
        )

    typical_daily_error = float(
        item_error["typical_daily_error"].iloc[0]
    )

    item_group = str(item_error["group"].iloc[0])

    # Calculate safety stock and reorder point
    safety_stock = (
        1.65 * typical_daily_error * math.sqrt(lead_time_days)
    )

    reorder_point = expected_demand_lead_time + safety_stock

    # Calculate how many days the current stock will last
    if avg_daily_forecast > 0:
        days_of_stock_left = current_stock / avg_daily_forecast
    else:
        days_of_stock_left = 999.0 if current_stock > 0 else 0.0

    # Calculate the required order quantity
    total_window_days = lead_time_days + review_days

    if len(daily_forecasts) >= total_window_days:
        expected_demand_window = float(
            np.sum(daily_forecasts[:total_window_days])
        )
    else:
        missing_days = total_window_days - len(daily_forecasts)

        expected_demand_window = (
            float(np.sum(daily_forecasts))
            + missing_days * avg_daily_forecast
        )

    raw_order_qty = (
        expected_demand_window + safety_stock - current_stock
    )

    order_quantity = max(0, math.ceil(raw_order_qty))

    # Classify stockout risk
    if days_of_stock_left < lead_time_days:
        risk = "High"
    elif days_of_stock_left < lead_time_days + 3:
        risk = "Medium"
    else:
        risk = "Low"

    # Return the reorder results
    return {
        "item_id": item_id,
        "group": item_group,
        "forecast_model_used": forecast_source,
        "current_stock": current_stock,
        "lead_time_days": lead_time_days,
        "average_daily_forecast": round(avg_daily_forecast, 2),
        "expected_demand_in_lead_time": round(expected_demand_lead_time, 2),
        "typical_daily_error": round(typical_daily_error, 2),
        "safety_stock": round(safety_stock, 2),
        "reorder_point": round(reorder_point, 2),
        "days_of_stock_left": round(days_of_stock_left, 1),
        "order_quantity": int(order_quantity),
        "risk": risk
    }


# Keep the old function name working
calculate_reorder_plan = reorder_advice
'''


# Save the updated module
with open(reorder_py_path, "w") as file:
    file.write(updated_reorder_code)

print("[SAVED] Updated reorder.py:", reorder_py_path)


# Import and reload the module
if EXPORT_DIR not in sys.path:
    sys.path.insert(0, EXPORT_DIR)

import reorder
importlib.reload(reorder)

from reorder import reorder_advice, calculate_reorder_plan

print("[OK] Both functions imported successfully.")


# Select one sample item for testing
sample_item = item_error_spread["item_id"].iloc[0]

print("--- Running Verification Tests ---")


# Test 1: Normal reorder case
res_normal = reorder_advice(
    sample_item,
    current_stock=5,
    lead_time_days=5,
    h7_preds_df=h7_val_preds,
    h28_preds_df=h28_val_preds,
    error_spread_df=item_error_spread
)

print(
    f"[PASS] Test 1 (Normal Case): "
    f"model={res_normal['forecast_model_used']}, "
    f"order_qty={res_normal['order_quantity']}, "
    f"risk={res_normal['risk']}"
)


# Test 2: Zero stock
res_zero_stock = reorder_advice(
    sample_item,
    current_stock=0,
    lead_time_days=5,
    h7_preds_df=h7_val_preds,
    h28_preds_df=h28_val_preds,
    error_spread_df=item_error_spread
)

print(
    f"[PASS] Test 2 (Zero Stock): "
    f"days_left={res_zero_stock['days_of_stock_left']}, "
    f"order_qty={res_zero_stock['order_quantity']}, "
    f"risk={res_zero_stock['risk']}"
)


# Test 3: Zero demand
zero_demand_df = pd.DataFrame({
    "item_id": [sample_item] * 14,
    "date": pd.date_range("2016-03-28", periods=14),
    "forecast": [0.0] * 14
})

res_zero_demand = reorder_advice(
    sample_item,
    current_stock=10,
    lead_time_days=5,
    h7_preds_df=zero_demand_df,
    h28_preds_df=zero_demand_df,
    error_spread_df=item_error_spread
)

print(
    f"[PASS] Test 3 (Zero Demand): "
    f"days_left={res_zero_demand['days_of_stock_left']}, "
    f"order_qty={res_zero_demand['order_quantity']}, "
    f"risk={res_zero_demand['risk']}"
)


# Test 4: Negative stock must be rejected
try:
    reorder_advice(
        sample_item,
        current_stock=-5,
        lead_time_days=5,
        h7_preds_df=h7_val_preds,
        h28_preds_df=h28_val_preds,
        error_spread_df=item_error_spread
    )

    print("[FAIL] Test 4: Negative stock was not rejected.")

except ValueError as e:
    print(f"[PASS] Test 4 (Negative Stock): {e}")


# Test 5: Zero lead time must be rejected
try:
    reorder_advice(
        sample_item,
        current_stock=10,
        lead_time_days=0,
        h7_preds_df=h7_val_preds,
        h28_preds_df=h28_val_preds,
        error_spread_df=item_error_spread
    )

    print("[FAIL] Test 5: Zero lead time was not rejected.")

except ValueError as e:
    print(f"[PASS] Test 5 (Zero Lead Time): {e}")


# Test 6: Invalid item ID must be rejected
try:
    reorder_advice(
        "INVALID_ITEM_999",
        current_stock=10,
        lead_time_days=5,
        h7_preds_df=h7_val_preds,
        h28_preds_df=h28_val_preds,
        error_spread_df=item_error_spread
    )

    print("[FAIL] Test 6: Invalid item ID was not rejected.")

except ValueError as e:
    print(f"[PASS] Test 6 (Invalid Item ID): {e}")


# Verify the saved file
print("--- Final File Verification ---")

for filename in ["reorder.py"]:
    filepath = os.path.join(EXPORT_DIR, filename)

    if os.path.isfile(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        print(f"[SAVED] {filename} ({size_kb:.2f} KB)")
    else:
        print(f"[MISSING] {filename}")

print("Verification execution completed.")

[SAVED] Updated reorder.py: /content/drive/MyDrive/DemandLens/Data/processed/reorder.py
[OK] Both functions imported successfully.
--- Running Verification Tests ---
[PASS] Test 1 (Normal Case): model=H7, order_qty=12, risk=Medium
[PASS] Test 2 (Zero Stock): days_left=0.0, order_qty=17, risk=High
[PASS] Test 3 (Zero Demand): days_left=999.0, order_qty=0, risk=Low
[PASS] Test 4 (Negative Stock): Invalid current_stock: must be a number >= 0.
[PASS] Test 5 (Zero Lead Time): Invalid lead_time_days: must be a positive integer.
[PASS] Test 6 (Invalid Item ID): Item 'INVALID_ITEM_999' not found in H7 forecast data.
--- Final File Verification ---
[SAVED] reorder.py (4.11 KB)
Verification execution completed.
